# NB07 · Altas potencialmente duplicadas

Una regla reproducible para decidir si una alta nueva es un duplicado de algo que ya está en el catálogo, calibrada **solo** con las 14 altas de desarrollo y congelada antes de abrir el conjunto de evaluación.

### Lo que se hereda y no se vuelve a decidir

| | |
|---|---|
| Motor · colección | Qdrant (R03) · `aurum_catalogo__gemini_embedding_2__A4__768` (NB04) |
| `ef` de consulta | **32** (R04, NB06) — la misma colección de producción, no el oráculo exacto |
| Interfaz | `BuscadorVectorial`, igual que NB05/NB06 |

### Las decisiones de este notebook (`config.yaml` → `nb07_duplicados`)

| | Decidido | Por qué |
|---|---|---|
| **D19** | Los candidatos salen de Qdrant tal como está *ahora*, antes de que NB08 aplique ningún evento | La base vectorial debe seguir siendo el mecanismo de generación de candidatos (enunciado §4.2); recalcular en local no lo prueba |
| **D20** | Señales: `score_top1`, margen (top1−top2), marca, color | Léxico de título descartado -el embedding ya debería cubrir títulos reordenados-; marca y color usan el payload, no el `text` codificado |
| **D21** | Dos caminos en OR: `score ≥ umbral_texto_solo ∧ margen ≥ margen_minimo` **o** `score ≥ umbral_texto_corroborado ∧ (marca ∨ color)` | El color nunca bloquea un duplicado por sí solo -"mismo producto, otra talla o color" puede contar como duplicado-; el OR entre marca y color evita que el 37,4%/4,4% de productos sin esos campos bloqueen el segundo camino |
| **D22** 🚨 | Maximizar recall con como mucho **2 falsos positivos** de los 7 negativos de desarrollo | Fijado **antes** de ver la curva P/R -si se fija después, deja de ser una restricción y pasa a describir el resultado, mismo patrón que D16-. Los falsos negativos degradan el top-10 de búsqueda con variantes redundantes sin que nada vuelva a auditar el catálogo publicado |

`umbral_texto_solo` (antes "τ_alto") es el listón para fiarse del texto **solo**, sin corroboración; `umbral_texto_corroborado` (antes "τ_bajo") es el listón, más bajo, para cuando marca o color respaldan la similitud; `margen_minimo` (antes "δ") es cuánto tiene que ganarle el top-1 al top-2 para no ser una foto-finish.

> ⚠️ El Camino 2 (corroborado) no tiene ni un ejemplo que lo active en solitario dentro de las 14 altas de desarrollo -los 7 positivos son reenvíos casi literales que ya caen en el Camino 1-. `umbral_texto_corroborado` se fija por criterio razonado, no por barrido: es una limitación declarada, no una carencia oculta.

In [1]:
# 📄 DATOS · 📚 altas_desarrollo.csv (14) + altas_evaluacion.csv (14)
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("..") / "src"))

from dotenv import load_dotenv

from aurum.busqueda import BuscadorVectorial
from aurum.datos import load_csv
from aurum.duplicados import (
    barrido_umbrales,
    calcular_senales,
    elegir_punto_operacion,
    resultados_duplicados,
)
from aurum.embeddings import GeminiEncoder, encode_corpus, truncate_dim
from aurum.graficas import plot_duplicate_threshold_sweep
from aurum.motores import CATALOG_PREFIX, catalog_collection_name
from aurum.motores.qdrant import QdrantStore

load_dotenv(Path("..") / ".env")
DATA = Path("..") / "data"
CACHE = Path("..") / "artifacts" / "embeddings"
altas_desarrollo = load_csv(DATA / "altas_desarrollo.csv")
altas_evaluacion = load_csv(DATA / "altas_evaluacion.csv")

MODELO, CONTRATO, PLANTILLA = "gemini-embedding-2", "sin_contrato", "A4"
DIM = 768
COLECCION = catalog_collection_name(model=MODELO, template=PLANTILLA, dim=DIM)
EF = 32  # R04 (NB06): el punto de operación de producción, no el oráculo exacto
MAX_FP = 2  # D22

print(f"desarrollo : {len(altas_desarrollo)} altas ({altas_desarrollo['is_duplicate'].sum()} duplicados)")
print(f"evaluacion : {len(altas_evaluacion)} altas, sin etiqueta")
print(f"coleccion  : {COLECCION} · ef={EF}")

c:\Users\asus\Master\modulos\modulo10_bbdd\practica\AURUM_MARKET\aurum-market-catalog\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


desarrollo : 14 altas (7 duplicados)
evaluacion : 14 altas, sin etiqueta
coleccion  : aurum_catalogo__gemini_embedding_2__A4__768 · ef=32


## A · Conexión a Qdrant

La misma colección de NB04-NB06, todavía sin las mutaciones de NB08 (D19). `codificar_consulta` no lleva instrucción de tarea -`CONTRATO="sin_contrato"`, D10- así que `kind="document"` aquí es solo trazabilidad de caché, no cambia el vector.

In [2]:
_encoder = GeminiEncoder(
    api_key=os.environ.get("GEMINI_API_KEY"), model_id=MODELO,
    native_dim=3072, window=8192,
)


def codificar_consulta(texto: str):
    codificado = encode_corpus(
        _encoder, [texto], corpus_id="altas_duplicados",
        kind="document", contract=CONTRATO, batch_size=1, cache_dir=CACHE,
    )
    return truncate_dim(codificado.vectors, DIM)[0]


almacen = QdrantStore(
    collection=COLECCION,
    url=os.environ.get("AURUM_QDRANT_URL", "http://localhost:6333"),
    api_key=os.environ.get("AURUM_QDRANT_API_KEY"),
    prefix=CATALOG_PREFIX,
    timeout=30,
)
buscador = BuscadorVectorial(almacen, codificar_consulta, top_k=2, ef=EF)
print(f"puntos en la coleccion: {almacen.count():,}".replace(",", ".") +
      f" · indice al dia: {almacen.index_ready()}")

puntos en la coleccion: 15.000 · indice al dia: True


## B · Señales de DESARROLLO (D20)

Para cada una de las 14 altas: su top-2 en la colección, y las cuatro señales frente al top-1. `margen = +∞` señalaría que no hubo segundo candidato -no debería pasar con 15.000 puntos, pero `SenalesDuplicado.margen` lo trata como la máxima confianza posible, no como cero-.

In [3]:
senales_desarrollo = calcular_senales(buscador, altas_desarrollo)

tabla_senales = pd.DataFrame([
    {
        "incoming_id": s.incoming_id,
        "matched_product_id": s.matched_product_id,
        "score_top1": round(s.score_top1, 4),
        "score_top2": round(s.score_top2, 4),
        "margen": round(s.margen, 4),
        "marca_coincide": s.marca_coincide,
        "color_coincide": s.color_coincide,
        "is_duplicate (real)": s.is_duplicate,
    }
    for s in senales_desarrollo
])
tabla_senales.style.hide(axis="index")

incoming_id,matched_product_id,score_top1,score_top2,margen,marca_coincide,color_coincide,is_duplicate (real)
DEV-DUP-001,B000G3T55M,0.984000,0.982600,0.001400,True,True,True
DEV-DUP-002,B07NV4L2W5,0.951900,0.919500,0.032400,True,True,True
DEV-DUP-003,B00BEFAR80,0.840200,0.783500,0.056700,True,None,True
DEV-DUP-004,B076HKFZ8N,0.983800,0.917800,0.066000,True,True,True
DEV-DUP-005,B07JYHSK27,0.979500,0.631600,0.347900,True,None,True
DEV-DUP-006,B07N379P73,0.880100,0.740900,0.139200,True,True,True
DEV-DUP-007,B077FZDNJ2,0.980900,0.942600,0.038300,True,True,True
DEV-NEW-001,B002VW5ZZU,0.715900,0.658100,0.057700,False,False,False
DEV-NEW-002,B0716LSWRG,0.597300,0.597300,0.000100,False,None,False
DEV-NEW-003,B07H9CLH2Q,0.588100,0.556100,0.032000,False,None,False


## C · El barrido (D21)

La rejilla de `umbral_texto_solo`/`umbral_texto_corroborado` sale del rango de `score_top1` **observado** en estas 14 altas, no de un valor de libro -no hay forma de saber de antemano en qué rango cae la similitud de `gemini-embedding-2` para este catálogo-. `margen_minimo` barre desde 0 hasta el margen máximo observado.

In [4]:
scores_observados = [s.score_top1 for s in senales_desarrollo]
margenes_observados = [s.margen for s in senales_desarrollo if s.margen != float("inf")]
SCORE_MIN, SCORE_MAX = min(scores_observados), max(scores_observados)
MARGEN_MAX = max(margenes_observados) if margenes_observados else 0.1

VALORES_UMBRAL_TEXTO_SOLO = sorted(set(round(v, 4) for v in np.linspace(SCORE_MIN, SCORE_MAX, 9)))
VALORES_UMBRAL_TEXTO_CORROBORADO = sorted(set(round(v, 4) for v in np.linspace(SCORE_MIN, SCORE_MAX, 9)))
VALORES_MARGEN_MINIMO = sorted(set(round(v, 4) for v in np.linspace(0.0, MARGEN_MAX, 6)))

barrido = barrido_umbrales(
    senales_desarrollo,
    valores_umbral_texto_solo=VALORES_UMBRAL_TEXTO_SOLO,
    valores_margen_minimo=VALORES_MARGEN_MINIMO,
    valores_umbral_texto_corroborado=VALORES_UMBRAL_TEXTO_CORROBORADO,
)
print(f"combinaciones evaluadas: {len(barrido)} "
      f"(umbral_texto_solo/umbral_texto_corroborado en [{SCORE_MIN:.3f}, {SCORE_MAX:.3f}], "
      f"margen_minimo en [0, {MARGEN_MAX:.3f}])")
# recall: de los 7 duplicados reales, cuántos atrapa la regla (1.0 = todos)
# precision: de lo que la regla marca duplicado, cuánto lo era de verdad
barrido.sort_values(["recall", "fp"], ascending=[False, True]).head(10).style.hide(axis="index")

combinaciones evaluadas: 216 (umbral_texto_solo/umbral_texto_corroborado en [0.588, 0.984], margen_minimo en [0, 0.348])


umbral_texto_solo,margen_minimo,umbral_texto_corroborado,tp,fp,fn,tn,precision,recall,f1
0.736600,0.000000,0.687100,7,0,0,7,1.000000,1.000000,1.000000
0.736600,0.069600,0.687100,7,0,0,7,1.000000,1.000000,1.000000
0.736600,0.139200,0.687100,7,0,0,7,1.000000,1.000000,1.000000
0.736600,0.208700,0.687100,7,0,0,7,1.000000,1.000000,1.000000
0.736600,0.278300,0.687100,7,0,0,7,1.000000,1.000000,1.000000
0.736600,0.347900,0.687100,7,0,0,7,1.000000,1.000000,1.000000
0.786100,0.000000,0.687100,7,0,0,7,1.000000,1.000000,1.000000
0.786100,0.000000,0.736600,7,0,0,7,1.000000,1.000000,1.000000
0.786100,0.069600,0.687100,7,0,0,7,1.000000,1.000000,1.000000
0.786100,0.069600,0.736600,7,0,0,7,1.000000,1.000000,1.000000


## D · El punto de operación (D22)

De las combinaciones con como mucho 2 falsos positivos, la de mayor recall -empate a recall y fp, gana el F1 más alto-. La tabla sale completa, con `cumple_d22` y `elegido_r05`, para poder contradecir la elección mirando también lo que no ganó.

In [5]:
tabla_operacion = elegir_punto_operacion(barrido, max_fp=MAX_FP)

figura_barrido = plot_duplicate_threshold_sweep(
    tabla_operacion, max_fp=MAX_FP,
    subtitle=f"{len(tabla_operacion)} combinaciones · 7 positivos + 7 negativos de desarrollo",
)
figura_barrido.show()

In [6]:
ganadora = tabla_operacion[tabla_operacion["elegido_r05"]]
if ganadora.empty:
    raise RuntimeError(
        "Ninguna combinacion del barrido cumple D22 (fp <= "
        f"{MAX_FP}): amplia VALORES_UMBRAL_TEXTO_SOLO/VALORES_UMBRAL_TEXTO_CORROBORADO/"
        "VALORES_MARGEN_MINIMO."
    )
fila = ganadora.iloc[0]
UMBRAL_TEXTO_SOLO = float(fila["umbral_texto_solo"])
MARGEN_MINIMO_BARRIDO = float(fila["margen_minimo"])
UMBRAL_TEXTO_CORROBORADO = float(fila["umbral_texto_corroborado"])

print("R05 -lo que devuelve el barrido-:")
print(f"  umbral_texto_solo        = {UMBRAL_TEXTO_SOLO}")
print(f"  margen_minimo (barrido)  = {MARGEN_MINIMO_BARRIDO}")
print(f"  umbral_texto_corroborado = {UMBRAL_TEXTO_CORROBORADO}")
print(f"  medido   : precision={fila['precision']:.3f} · recall={fila['recall']:.3f} "
      f"· f1={fila['f1']:.3f} · fp={int(fila['fp'])}/7")

R05 -lo que devuelve el barrido-:
  umbral_texto_solo        = 0.7366
  margen_minimo (barrido)  = 0.0
  umbral_texto_corroborado = 0.6871
  medido   : precision=1.000 · recall=1.000 · f1=1.000 · fp=0/7


## D.1 · Override manual de `margen_minimo`

El barrido devolvió `margen_minimo=0.0` no porque sea el mejor valor, sino porque **empata** con cualquier otro candidato -las 7 altas positivas tienen `marca_coincide=True` y ya superan `umbral_texto_corroborado` por sí solas, así que el Camino 2 las atrapa sin necesitar el margen (ver `aviso_margen_minimo_cero` en `config.yaml`)-. Subir `margen_minimo` no cuesta nada medible aquí, y añade la protección real que el margen fue pensado para dar.

El único dato que tenemos para elegir un valor es el margen más alto visto entre los **negativos** (0,0577, `DEV-NEW-001`) -el "ruido" que existe aunque no haya ningún duplicado real-. `margen_minimo=0.1` queda por encima de ese ruido sin exigir tanto como para que el Camino 1 dependa casi siempre del Camino 2 (con 0,2 solo `DEV-DUP-005` lo pasaría por sí sola).

In [7]:
MARGEN_MINIMO = 0.1  # override manual (D21): por encima del ruido de
# negativos (max observado 0.0577, DEV-NEW-001), decisión razonada, no
# resultado del barrido -que era indiferente a este valor-.

print(f"margen_minimo: barrido={MARGEN_MINIMO_BARRIDO} · usado={MARGEN_MINIMO} (override manual)")

margen_minimo: barrido=0.0 · usado=0.1 (override manual)


## E · 🚨 CONGELA — a partir de aquí, `altas_evaluacion.csv`

El protocolo es explícito: barrer, elegir, dejarlo registrado, **congelar**, y solo entonces abrir el conjunto de evaluación. `UMBRAL_TEXTO_SOLO`/`MARGEN_MINIMO`/`UMBRAL_TEXTO_CORROBORADO` no se vuelven a tocar de aquí en adelante.

In [8]:
senales_evaluacion = calcular_senales(buscador, altas_evaluacion, etiquetas_col=None)

resultados = resultados_duplicados(
    senales_evaluacion,
    umbral_texto_solo=UMBRAL_TEXTO_SOLO,
    margen_minimo=MARGEN_MINIMO,
    umbral_texto_corroborado=UMBRAL_TEXTO_CORROBORADO,
)

destino = Path("..") / "resultados" / "resultados_duplicados.csv"
resultados.to_csv(destino, index=False)
print(f"Escrito {destino} · {len(resultados)} filas · "
      f"{int(resultados['predicted_duplicate'].sum())} duplicados detectados")
resultados.style.hide(axis="index")

Escrito ..\resultados\resultados_duplicados.csv · 14 filas · 7 duplicados detectados


incoming_id,predicted_duplicate,matched_product_id,score
EVAL-DUP-001,True,B081JP8CC6,0.938768
EVAL-DUP-002,True,B07GWRF23V,0.916517
EVAL-DUP-003,True,B07S7B3SN2,0.915375
EVAL-DUP-004,True,8417441271,0.923077
EVAL-DUP-005,True,B00JOH9FRO,0.938776
EVAL-DUP-006,True,B01JADTEKE,0.979972
EVAL-DUP-007,True,B01M0G3WP0,0.943956
EVAL-NEW-001,False,,0.678005
EVAL-NEW-002,False,,0.664018
EVAL-NEW-003,False,,0.684486


## F · Lo que este notebook no puede afirmar, y por qué

| Limitación | Por qué |
|---|---|
| El Camino 2 (marca/color) no está calibrado empíricamente | Ningún ejemplo de las 14 altas de desarrollo lo activa en solitario -los 7 positivos ya caen en el Camino 1-. `τ_bajo` es una decisión razonada, no medida |
| El corroborador de marca no se activa en el 4,4% del catálogo | `brand` vacío en 658/15.000 productos (verificado sobre `catalogo_productos.csv`) |
| El corroborador de color no se activa en el 37,4% del catálogo | Mismo motivo, sobre `color` |
| La rejilla de umbrales/margen se calibra sobre 14 ejemplos | Cualquier cifra de precisión con más de un decimal es más precisa de lo que 7 negativos pueden sostener -por eso D22 se expresa en cuenta de falsos positivos, no en un umbral de precisión- |